In [7]:
import os

In [8]:
os.chdir('../')

In [9]:
os.getcwd()

'/Users/uw-user/Library/CloudStorage/GoogleDrive-vvithurshan@gmail.com/My Drive/Artificial-Intelligence-2025/Machine-Learning/PW-Skills/Projects/Chiken-Disease-Classification'

In [11]:
import sys
print("\n".join(sys.path))

/Users/uw-user/Library/CloudStorage/GoogleDrive-vvithurshan@gmail.com/My Drive/Artificial-Intelligence-2025/Machine-Learning/PW-Skills/Projects/Chiken-Disease-Classification/venv/lib/python311.zip
/Users/uw-user/Library/CloudStorage/GoogleDrive-vvithurshan@gmail.com/My Drive/Artificial-Intelligence-2025/Machine-Learning/PW-Skills/Projects/Chiken-Disease-Classification/venv/lib/python3.11
/Users/uw-user/Library/CloudStorage/GoogleDrive-vvithurshan@gmail.com/My Drive/Artificial-Intelligence-2025/Machine-Learning/PW-Skills/Projects/Chiken-Disease-Classification/venv/lib/python3.11/lib-dynload

/Users/uw-user/Library/CloudStorage/GoogleDrive-vvithurshan@gmail.com/My Drive/Artificial-Intelligence-2025/Machine-Learning/PW-Skills/Projects/Chiken-Disease-Classification/venv/lib/python3.11/site-packages


In [10]:
import tensorflow as tf

In [11]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

[2025-07-25 18:33:23,667: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]


In [12]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvalutionConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int

In [21]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [22]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):

            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)
            create_directories([self.config.artifacts_root])

    def get_validation_config(self) -> EvalutionConfig:
        eval_config = EvalutionConfig(
                path_of_model = "artifacts/training/model.h5",
                training_data = "artifacts/data_ingestion/Chicken-fecal-images",
                all_params = self.params,
                params_image_size = self.params.IMAGE_SIZE,
                params_batch_size = self.params.BATCH_SIZE
          )
        return eval_config
        

In [23]:
from urllib.parse import urlparse

In [24]:
class Evaluation:
    def __init__(self, config: EvalutionConfig):
        self.config = config

    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split = 0.30,
        )

        dataflow_kwargs = dict(
            target_size = self.config.params_image_size[: -1],
            batch_size = self.config.params_batch_size,
            interpolation = 'bilinear'
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory = self.config.training_data,
            subset = 'validation',
            shuffle = False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    
    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data = scores)


In [25]:
try:
    config = ConfigurationManager()
    val_config = config.get_validation_config()
    evaluation = Evaluation(val_config)
    evaluation.evaluation()
    evaluation.save_score()
except Exception as e:
    raise e

[2025-07-25 18:56:09,479: INFO: common: YAML file loaded successfully: config/config.yaml]
[2025-07-25 18:56:09,481: INFO: common: YAML file loaded successfully: params.yaml]
[2025-07-25 18:56:09,482: INFO: common: Created Directory at: artifacts]
[2025-07-25 18:56:09,643: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 116 images belonging to 2 classes.


/Users/uw-user/Library/CloudStorage/GoogleDrive-vvithurshan@gmail.com/My Drive/Artificial-Intelligence-2025/Machine-Learning/PW-Skills/Projects/Chiken-Disease-Classification/venv/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


8/8 ━━━━━━━━━━━━━━━━━━━━ 45s 6s/step - accuracy: 0.7504 - loss: 2.1517
[2025-07-25 18:56:55,042: INFO: common: json file saved at: scores.json]
